# Sales Prediction Using Machine Learning

This project predicts product sales from advertising spend on TV, Radio, and Newspaper.

The workflow includes data loading, data understanding, exploratory analysis, feature selection, model training, model comparison, cross-validation, feature importance, and a final prediction function.

## 1. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path(".matplotlib-cache").resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## 2. Load Dataset

In [ ]:
def load_sales_data():
    possible_paths = [
        Path("input.csv"),
        Path("/content/input.csv"),
        Path("/content/drive/MyDrive/project/input.csv"),
    ]

    for path in possible_paths:
        if path.exists():
            print(f"Loaded dataset from: {path}")
            return pd.read_csv(path)

    raise FileNotFoundError(
        "Dataset not found. Place input.csv in this project folder, upload it to /content/input.csv, "
        "or keep it at /content/drive/MyDrive/project/input.csv in Colab."
    )

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

df = load_sales_data()
df.head()

## 3. Data Understanding

In [ ]:
print("Dataset shape:", df.shape)
print("
Columns:", list(df.columns))
df.info()

In [ ]:
df.describe()

In [ ]:
missing_percent = df.isnull().sum() * 100 / len(df)
missing_percent

In [ ]:
duplicate_count = df.duplicated().sum()
print("Duplicate rows:", duplicate_count)

## 4. Exploratory Data Analysis

In [ ]:
numeric_columns = ["TV", "Radio", "Newspaper", "Sales"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()

for index, column in enumerate(numeric_columns):
    sns.histplot(df[column], kde=True, ax=axes[index])
    axes[index].set_title(f"Distribution of {column}")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()

for index, column in enumerate(numeric_columns):
    sns.boxplot(x=df[column], ax=axes[index])
    axes[index].set_title(f"Boxplot of {column}")

plt.tight_layout()
plt.show()

In [ ]:
sns.pairplot(
    df,
    x_vars=["TV", "Radio", "Newspaper"],
    y_vars="Sales",
    height=5,
    kind="scatter"
)
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="Blues", fmt=".2f")
plt.title("Feature Correlation Heatmap")
plt.show()

## 5. Feature Selection

In [ ]:
X = df[["TV", "Radio", "Newspaper"]]
y = df["Sales"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=RANDOM_STATE
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

## 6. Simple Linear Regression Baseline

In [ ]:
X_tv = df[["TV"]]
y_tv = df["Sales"]

X_tv_train, X_tv_test, y_tv_train, y_tv_test = train_test_split(
    X_tv,
    y_tv,
    test_size=0.3,
    random_state=100
)

simple_lr = LinearRegression()
simple_lr.fit(X_tv_train, y_tv_train)
y_tv_pred = simple_lr.predict(X_tv_test)

simple_results = {
    "Model": "Simple Linear Regression (TV only)",
    "R2": r2_score(y_tv_test, y_tv_pred),
    "MAE": mean_absolute_error(y_tv_test, y_tv_pred),
    "RMSE": np.sqrt(mean_squared_error(y_tv_test, y_tv_pred)),
}

simple_results

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(X_tv_train["TV"], y_tv_train, alpha=0.75, label="Training data")
plt.plot(
    X_tv_train["TV"],
    simple_lr.predict(X_tv_train),
    color="red",
    label="Regression line"
)
plt.xlabel("TV Advertising Spend")
plt.ylabel("Sales")
plt.title("Simple Linear Regression: TV vs Sales")
plt.legend()
plt.show()

## 7. Train and Compare Multiple Models

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Lasso Regression": Lasso(alpha=0.01, max_iter=10000),
    "Random Forest": RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
}

results = []
trained_models = {}

for model_name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    results.append({
        "Model": model_name,
        "R2": r2_score(y_test, y_pred),
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
    })

    trained_models[model_name] = model

results_df = pd.DataFrame(results).sort_values(by="RMSE")
results_df

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=results_df, x="RMSE", y="Model", palette="viridis")
plt.title("Model Comparison by RMSE")
plt.xlabel("RMSE lower is better")
plt.ylabel("")
plt.show()

## 8. Cross-Validation

In [ ]:
cv_results = []

for model_name, model in models.items():
    scores = cross_val_score(model, X, y, cv=5, scoring="r2")
    cv_results.append({
        "Model": model_name,
        "Mean CV R2": scores.mean(),
        "Std CV R2": scores.std(),
    })

cv_results_df = pd.DataFrame(cv_results).sort_values(by="Mean CV R2", ascending=False)
cv_results_df

## 9. Select Best Model

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]
best_predictions = best_model.predict(X_test)

print("Best model:", best_model_name)
print("R2 Score:", r2_score(y_test, best_predictions))
print("MAE:", mean_absolute_error(y_test, best_predictions))
print("RMSE:", np.sqrt(mean_squared_error(y_test, best_predictions)))

In [ ]:
comparison_df = pd.DataFrame({
    "Actual Sales": y_test.values,
    "Predicted Sales": best_predictions,
})
comparison_df["Error"] = comparison_df["Actual Sales"] - comparison_df["Predicted Sales"]
comparison_df.head(10)

In [ ]:
plt.figure(figsize=(7, 6))
sns.scatterplot(x=y_test, y=best_predictions)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color="red")
plt.xlabel("Actual Sales")
plt.ylabel("Predicted Sales")
plt.title(f"Actual vs Predicted Sales: {best_model_name}")
plt.show()

In [ ]:
residuals = y_test - best_predictions

plt.figure(figsize=(8, 5))
sns.histplot(residuals, kde=True)
plt.title("Residual Distribution")
plt.xlabel("Prediction Error")
plt.show()

## 10. Feature Importance

In [ ]:
feature_names = X.columns

if hasattr(best_model, "coef_"):
    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": best_model.coef_,
    }).sort_values(by="Importance", key=abs, ascending=False)
elif hasattr(best_model, "feature_importances_"):
    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": best_model.feature_importances_,
    }).sort_values(by="Importance", ascending=False)
else:
    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": np.nan,
    })

importance_df

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=importance_df, x="Importance", y="Feature", palette="mako")
plt.title(f"Feature Importance: {best_model_name}")
plt.show()

## 11. Final Prediction Function

In [ ]:
def predict_sales(tv, radio, newspaper):
    input_data = pd.DataFrame(
        [[tv, radio, newspaper]],
        columns=["TV", "Radio", "Newspaper"]
    )
    prediction = best_model.predict(input_data)[0]
    return round(prediction, 2)

example_prediction = predict_sales(tv=200, radio=30, newspaper=20)
print("Predicted Sales:", example_prediction)

## 12. Conclusion

The improved workflow compares several regression models instead of relying on one model only. The model with the lowest RMSE is selected as the final model.

In most runs on this advertising dataset, TV and Radio are stronger predictors of Sales than Newspaper. The final prediction function can estimate expected sales from new advertising budget values.